# BioMini Phase I Educational Materials

## Notebook 05 — From Biology to Feature Engineering
### Biological Object에서 Machine-Learning Representation으로

Notebook 04에서는 biological domain object와 analysis logic을 분리했습니다.

이번 Notebook에서는 그 분석 결과를 machine-learning model이 사용할 수 있는
**numerical representation**으로 바꾸는 방법을 다룹니다.

핵심 흐름은 다음과 같습니다.

```text
Protein
   ↓
ProteinAnalyzer
   ↓
FeatureSet
   ↓
Sample
   ↓
Dataset
```

이 Notebook은 BioMini Phase I의 **Step 7 — FeatureSet / Sample / Dataset**을
교육용으로 재구성합니다.

---

## Learning Objectives

이 Notebook을 마치면 다음을 할 수 있어야 합니다.

1. biological object와 ML representation의 차이를 설명한다.
2. feature engineering의 역할을 설명한다.
3. `FeatureSet` abstraction이 왜 필요한지 이해한다.
4. feature name과 feature value를 함께 보존해야 하는 이유를 설명한다.
5. `Sample`과 `Dataset`의 역할을 구분한다.
6. 여러 Protein object를 tabular ML dataset으로 변환한다.
7. feature validation의 필요성을 설명한다.
8. target이 없는 dataset과 supervised-learning dataset의 차이를 이해한다.
9. canonical BioMini의 `FeatureSet`, `Sample`, `Dataset`과 교육용 구현을 비교한다.

## 1. Starting Point — ML은 Protein Object를 직접 사용하지 않는다

`Protein` object는 biological meaning을 잘 표현합니다.

예:

```text
Protein
- name
- sequence
- amino acids
```

하지만 scikit-learn 같은 일반적인 ML model은 이런 Python object를 직접 입력으로 사용하지 않습니다.

ML model이 기대하는 것은 보통 숫자로 이루어진 table 또는 matrix입니다.

예:

```text
length    molecular_weight    hydrophobic_fraction
   20          2395.7                0.40
   12          1450.3                0.58
   35          4020.1                0.31
```

따라서 다음 bridge가 필요합니다.

```text
Biological Object
        ↓
Scientific Analysis
        ↓
Numerical Features
```

## 2. 최소 Domain / Analyzer 준비

이번 Notebook은 독립 실행이 가능하도록 필요한 최소 class를 포함합니다.

In [ ]:
class ValidationError(ValueError):
    pass


class SequenceMolecule:
    def __init__(self, name: str, sequence: str):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("name must be a non-empty string")

        if not isinstance(sequence, str):
            raise ValidationError("sequence must be a string")

        self.name = name.strip()
        self.sequence = "".join(sequence.split()).upper()

    def __len__(self) -> int:
        return len(self.sequence)

In [ ]:
class AminoAcid:
    mass_dict = {
        "A": 71.0779, "R": 156.1857, "N": 114.1026, "D": 115.0874,
        "C": 103.1429, "Q": 128.1292, "E": 129.1140, "G": 57.0513,
        "H": 137.1393, "I": 113.1576, "L": 113.1576, "K": 128.1723,
        "M": 131.1960, "F": 147.1738, "P": 97.1152, "S": 87.0773,
        "T": 101.1039, "W": 186.2099, "Y": 163.1732, "V": 99.1310,
    }

    acceptable_codes = set(mass_dict)

    def __init__(self, code: str):
        normalized = code.upper()
        if normalized not in self.acceptable_codes:
            raise ValidationError(f"invalid amino acid: {normalized}")
        self.code = normalized

    def get_mass(self) -> float:
        return self.mass_dict[self.code]

In [ ]:
class Protein(SequenceMolecule):
    valid_codes = AminoAcid.acceptable_codes

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)

        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid protein code(s): {sorted(invalid_codes)}"
            )

        self.amino_acids = [
            AminoAcid(code)
            for code in self.sequence
        ]

In [ ]:
class ProteinAnalyzer:
    hydrophobic_codes = set("AILMFWVY")

    def __init__(self, protein: Protein):
        self.protein = protein

    def length(self) -> int:
        return len(self.protein)

    def molecular_weight(self) -> float:
        if not self.protein.amino_acids:
            return 0.0

        return 18.0153 + sum(
            amino_acid.get_mass()
            for amino_acid in self.protein.amino_acids
        )

    def hydrophobic_fraction(self) -> float:
        sequence = self.protein.sequence

        if not sequence:
            return 0.0

        count = sum(
            1
            for code in sequence
            if code in self.hydrophobic_codes
        )

        return count / len(sequence)

    def amino_acid_fraction_features(self) -> dict[str, float]:
        sequence = self.protein.sequence

        if not sequence:
            return {
                f"aa_{code}_fraction": 0.0
                for code in sorted(AminoAcid.acceptable_codes)
            }

        return {
            f"aa_{code}_fraction":
                sequence.count(code) / len(sequence)

            for code in sorted(
                AminoAcid.acceptable_codes
            )
        }

## 3. Handcrafted Protein Features

Phase I에서는 intentionally simple한 handcrafted feature를 사용합니다.

예:

```text
length
molecular_weight
hydrophobic_fraction
amino-acid fractions
```

이것은 최종 Scientific AI representation이 아닙니다.

중요한 것은 **biological object를 numerical representation으로 변환하는 interface**를 만드는 것입니다.

In [ ]:
protein = Protein(
    "protein1",
    "ACDEFGHIKLMNPQRSTVWY"
)

analyzer = ProteinAnalyzer(
    protein
)

print("length:", analyzer.length())
print("molecular weight:", analyzer.molecular_weight())
print("hydrophobic fraction:", analyzer.hydrophobic_fraction())

## 4. 단순 Dictionary로 시작하기

가장 간단하게는 feature를 dictionary로 표현할 수 있습니다.

In [ ]:
features = {
    "length": analyzer.length(),
    "molecular_weight": analyzer.molecular_weight(),
    "hydrophobic_fraction": analyzer.hydrophobic_fraction(),
}

features.update(
    analyzer.amino_acid_fraction_features()
)

print(features)

이 방식은 이미 꽤 유용합니다.

하지만 project가 커지면 다음 문제가 생깁니다.

```text
feature dictionary의 이름은 무엇인가?
feature value가 숫자가 아닌 경우는?
NaN이나 infinity가 들어오면?
빈 feature set은 허용할 것인가?
serialization은 어떻게 할 것인가?
```

따라서 plain dictionary를 하나의 명시적인 object로 감싸는 것이 좋습니다.

## 5. FeatureSet Abstraction

`FeatureSet`은 하나의 biological sample을 설명하는 numerical feature 묶음입니다.

canonical BioMini에서는 다음 조건을 사용합니다.

```text
name
- non-empty string

features
- non-empty dictionary
- key는 non-empty string
- value는 numeric
- value는 finite
```

In [ ]:
from math import isfinite
from numbers import Real


class FeatureSet:
    def __init__(
        self,
        name: str,
        features: dict[str, float]
    ):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError(
                "FeatureSet name must be a non-empty string"
            )

        if not isinstance(features, dict) or not features:
            raise ValidationError(
                "features must be a non-empty dictionary"
            )

        normalized_features = {}

        for key, value in features.items():
            if not isinstance(key, str) or not key.strip():
                raise ValidationError(
                    "feature names must be non-empty strings"
                )

            if isinstance(value, bool) or not isinstance(value, Real):
                raise ValidationError(
                    f"feature {key!r} must be numeric"
                )

            numeric_value = float(value)

            if not isfinite(numeric_value):
                raise ValidationError(
                    f"feature {key!r} must be finite"
                )

            normalized_features[
                key.strip()
            ] = numeric_value

        self.name = name.strip()
        self.features = normalized_features

    def get(
        self,
        feature_name: str
    ):
        return self.features[
            feature_name
        ]

    def __repr__(self) -> str:
        return (
            f"FeatureSet("
            f"name='{self.name}', "
            f"features={self.features})"
        )

In [ ]:
feature_set = FeatureSet(
    protein.name,
    features
)

print(feature_set)
print(
    "length:",
    feature_set.get("length")
)

## 6. ProteinAnalyzer → FeatureSet

이제 Analyzer가 numerical values만 계산하는 데 그치지 않고,
정형화된 `FeatureSet`을 만들어 주도록 할 수 있습니다.

In [ ]:
class ProteinAnalyzer:
    hydrophobic_codes = set("AILMFWVY")

    def __init__(self, protein: Protein):
        self.protein = protein

    def length(self) -> int:
        return len(self.protein)

    def molecular_weight(self) -> float:
        if not self.protein.amino_acids:
            return 0.0

        return 18.0153 + sum(
            amino_acid.get_mass()
            for amino_acid in self.protein.amino_acids
        )

    def hydrophobic_fraction(self) -> float:
        sequence = self.protein.sequence

        if not sequence:
            return 0.0

        count = sum(
            1
            for code in sequence
            if code in self.hydrophobic_codes
        )

        return count / len(sequence)

    def amino_acid_fraction_features(self) -> dict[str, float]:
        sequence = self.protein.sequence

        if not sequence:
            return {
                f"aa_{code}_fraction": 0.0
                for code in sorted(AminoAcid.acceptable_codes)
            }

        return {
            f"aa_{code}_fraction":
                sequence.count(code) / len(sequence)

            for code in sorted(
                AminoAcid.acceptable_codes
            )
        }

    def to_features(self) -> FeatureSet:
        features = {
            "length":
                self.length(),

            "molecular_weight":
                self.molecular_weight(),

            "hydrophobic_fraction":
                self.hydrophobic_fraction(),
        }

        features.update(
            self.amino_acid_fraction_features()
        )

        return FeatureSet(
            self.protein.name,
            features
        )

In [ ]:
protein = Protein(
    "protein2",
    "MKWVTFISLLFLFSSAYSR"
)

analyzer = ProteinAnalyzer(
    protein
)

feature_set = analyzer.to_features()

print(feature_set)

여기까지의 흐름은 다음과 같습니다.

```text
Protein
   ↓
ProteinAnalyzer
   ↓
FeatureSet
```

중요한 것은 `Protein` 자체와 numerical feature representation을 분리했다는 점입니다.

## 7. Why Feature Names Matter

단순 list:

```text
[20, 2395.7, 0.40]
```

는 어떤 값이 무엇을 의미하는지 쉽게 잃어버릴 수 있습니다.

반면:

```text
length = 20
molecular_weight = 2395.7
hydrophobic_fraction = 0.40
```

처럼 name을 유지하면 의미가 보존됩니다.

이는 나중에 model training과 inference에서 feature order를 일관되게 유지하는 데도 중요합니다.

## 8. Feature Validation 확인

In [ ]:
invalid_feature_sets = [
    (
        "empty features",
        {}
    ),
    (
        "non-numeric value",
        {"length": "twenty"}
    ),
    (
        "infinite value",
        {"length": float("inf")}
    ),
    (
        "bad feature name",
        {"": 1.0}
    ),
]

for label, bad_features in invalid_feature_sets:
    try:
        FeatureSet(
            "bad",
            bad_features
        )

    except ValidationError as exc:
        print(
            f"{label:20s} -> {exc}"
        )

## 9. Sample — FeatureSet과 Target 연결

ML에서는 feature뿐 아니라 target이 필요할 수 있습니다.

예:

```text
Protein features
   ↓
target = experimentally measured activity
```

이를 `Sample`로 표현합니다.

```text
Sample
├── FeatureSet
└── target
```

In [ ]:
class Sample:
    def __init__(
        self,
        feature_set: FeatureSet,
        target=None
    ):
        if not isinstance(
            feature_set,
            FeatureSet
        ):
            raise ValidationError(
                "feature_set must be a FeatureSet"
            )

        self.feature_set = feature_set
        self.target = target

In [ ]:
sample = Sample(
    feature_set,
    target=0.82
)

print(
    sample.feature_set.name
)

print(
    sample.target
)

`target=None`도 허용합니다.

이는 prediction-only sample이나 아직 label이 없는 sample을 표현하는 데 유용합니다.

In [ ]:
unlabeled_sample = Sample(
    feature_set
)

print(
    unlabeled_sample.target
)

## 10. Dataset — 여러 Sample을 하나로 묶기

ML training에는 여러 sample이 필요합니다.

따라서 다음 abstraction을 도입합니다.

```text
Dataset
 ├── Sample 1
 ├── Sample 2
 ├── Sample 3
 └── ...
```

In [ ]:
class DataError(ValueError):
    pass


class Dataset:
    def __init__(self):
        self.samples: list[Sample] = []

    def add(
        self,
        sample: Sample
    ):
        if not isinstance(sample, Sample):
            raise ValidationError(
                "dataset entries must be Sample objects"
            )

        self.samples.append(
            sample
        )

    def __len__(self):
        return len(
            self.samples
        )

In [ ]:
dataset = Dataset()

dataset.add(
    Sample(
        ProteinAnalyzer(
            Protein(
                "p1",
                "ACDEFG"
            )
        ).to_features(),
        target=0.20
    )
)

dataset.add(
    Sample(
        ProteinAnalyzer(
            Protein(
                "p2",
                "AILMFWVY"
            )
        ).to_features(),
        target=0.85
    )
)

print(
    "dataset size:",
    len(dataset)
)

## 11. Dataset을 Records로 변환하기

각 `Sample`을 row 형태로 바꾸면 tabular data를 만들 수 있습니다.

In [ ]:
def dataset_to_records(
    dataset: Dataset
):
    records = []

    for sample in dataset.samples:
        row = {
            "name":
                sample.feature_set.name,

            **sample.feature_set.features,
        }

        if sample.target is not None:
            row["target"] = (
                sample.target
            )

        records.append(
            row
        )

    return records

In [ ]:
records = dataset_to_records(
    dataset
)

for row in records:
    print(row)

## 12. pandas DataFrame으로 변환하기

scikit-learn과 연결하기 위해 pandas DataFrame이 매우 유용합니다.

In [ ]:
import pandas as pd

df = pd.DataFrame(
    records
)

df

이 시점에서 biological object는 완전히 numerical tabular representation으로 변환되었습니다.

```text
Protein
 ↓
ProteinAnalyzer
 ↓
FeatureSet
 ↓
Sample
 ↓
Dataset
 ↓
DataFrame
```

## 13. Dataset.to_dataframe()

이 변환을 Dataset method로 정리합니다.

In [ ]:
class Dataset:
    def __init__(self):
        self.samples: list[Sample] = []

    def add(
        self,
        sample: Sample
    ):
        if not isinstance(sample, Sample):
            raise ValidationError(
                "dataset entries must be Sample objects"
            )

        self.samples.append(
            sample
        )

    def __len__(self):
        return len(
            self.samples
        )

    def to_records(self):
        records = []

        for sample in self.samples:
            row = {
                "name":
                    sample.feature_set.name,

                **sample.feature_set.features,
            }

            if sample.target is not None:
                row["target"] = (
                    sample.target
                )

            records.append(
                row
            )

        return records

    def to_dataframe(self):
        return pd.DataFrame(
            self.to_records()
        )

## 14. 조금 더 큰 Protein Dataset 만들기

교육용 작은 dataset을 만들어 봅니다.

target은 실제 biological measurement가 아니라
ML pipeline을 연습하기 위한 synthetic value입니다.

In [ ]:
protein_data = [
    (
        "protein_A",
        "ACDEFGHIK",
        0.25
    ),
    (
        "protein_B",
        "AILMFWVY",
        0.92
    ),
    (
        "protein_C",
        "DEKRQN",
        0.10
    ),
    (
        "protein_D",
        "MKWVTFISLLFLFSSAYSR",
        0.78
    ),
    (
        "protein_E",
        "GGGGAAAAVVVV",
        0.65
    ),
]

dataset = Dataset()

for name, sequence, target in protein_data:
    protein = Protein(
        name,
        sequence
    )

    feature_set = (
        ProteinAnalyzer(
            protein
        ).to_features()
    )

    sample = Sample(
        feature_set,
        target=target
    )

    dataset.add(
        sample
    )

df = dataset.to_dataframe()

df

## 15. X와 y의 개념

Supervised learning에서는 일반적으로:

```text
X
= input features

y
= target
```

로 나눕니다.

이 변환을 직접 해 봅니다.

In [ ]:
X = df.drop(
    columns=[
        "name",
        "target"
    ]
)

y = df[
    "target"
]

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

In [ ]:
X.head()

In [ ]:
y

## 16. Dataset.to_xy()

canonical BioMini에서는 이 기능을 Dataset method로 제공합니다.

target column이 없는 dataset에서 `to_xy()`를 호출하면
`DataError`를 발생시킵니다.

In [ ]:
class Dataset:
    def __init__(self):
        self.samples: list[Sample] = []

    def add(
        self,
        sample: Sample
    ):
        if not isinstance(sample, Sample):
            raise ValidationError(
                "dataset entries must be Sample objects"
            )

        self.samples.append(
            sample
        )

    def __len__(self):
        return len(
            self.samples
        )

    def to_records(self):
        records = []

        for sample in self.samples:
            row = {
                "name":
                    sample.feature_set.name,

                **sample.feature_set.features,
            }

            if sample.target is not None:
                row["target"] = (
                    sample.target
                )

            records.append(
                row
            )

        return records

    def to_dataframe(self):
        return pd.DataFrame(
            self.to_records()
        )

    def to_xy(self):
        df = self.to_dataframe()

        if "target" not in df.columns:
            raise DataError(
                "dataset has no target"
            )

        X = df.drop(
            columns=[
                "name",
                "target"
            ]
        )

        y = df[
            "target"
        ]

        return X, y

In [ ]:
# Dataset class was redefined above to add to_xy().
# Rebuild the dataset so this instance uses the updated class definition.
dataset = Dataset()

for name, sequence, target in protein_data:
    protein = Protein(name, sequence)
    feature_set = ProteinAnalyzer(protein).to_features()
    dataset.add(Sample(feature_set, target=target))

X, y = dataset.to_xy()

print(
    X.shape,
    y.shape
)


## 17. Target이 없는 Dataset

이제 label이 없는 dataset을 만들어 봅니다.

In [ ]:
prediction_dataset = Dataset()

prediction_dataset.add(
    Sample(
        ProteinAnalyzer(
            Protein(
                "unknown_1",
                "ACDEFG"
            )
        ).to_features()
    )
)

print(
    prediction_dataset
    .to_dataframe()
)

In [ ]:
try:
    prediction_dataset.to_xy()

except DataError as exc:
    print(
        type(exc).__name__,
        ":",
        exc
    )

이것은 error handling의 중요한 예입니다.

`Dataset` 자체는 unlabeled sample을 가질 수 있습니다.

하지만 supervised-learning용 `X, y`를 요청했는데 target이 없다면
그 요청은 수행할 수 없습니다.

따라서:

```text
Dataset 존재 가능
하지만
to_xy() 수행 불가
```

라는 semantic을 명시적으로 표현합니다.

## 18. Feature Representation과 Biological Object를 분리하는 이유

같은 Protein도 여러 방식으로 표현할 수 있습니다.

Phase I:

```text
handcrafted features
```

Phase II:

```text
k-mer
one-hot
tokenization
embedding
foundation-model representation
```

즉:

```text
Protein object
```

자체를 바꾸지 않고 representation만 교체할 수 있어야 합니다.

이것이 `FeatureSet` layer를 따로 둔 중요한 이유입니다.

## 19. Self-Check

In [ ]:
protein = Protein(
    "self_check",
    "AILM"
)

feature_set = (
    ProteinAnalyzer(
        protein
    ).to_features()
)

assert (
    feature_set.get("length")
    == 4.0
)

assert (
    feature_set.get(
        "hydrophobic_fraction"
    )
    == 1.0
)

sample = Sample(
    feature_set,
    target=1.0
)

test_dataset = Dataset()
test_dataset.add(
    sample
)

assert (
    len(test_dataset)
    == 1
)

X, y = (
    test_dataset.to_xy()
)

assert (
    X.shape[0]
    == 1
)

assert (
    y.iloc[0]
    == 1.0
)

print(
    "Notebook 05 self-check: PASS"
)

## 20. Canonical BioMini와 비교

실제 BioMini package가 설치되어 있다면
현재 production implementation을 직접 사용할 수 있습니다.

In [ ]:
try:
    from biomini import (
        Protein as BioMiniProtein,
        ProteinAnalyzer as BioMiniProteinAnalyzer,
        Sample as BioMiniSample,
        Dataset as BioMiniDataset,
    )

    bm_protein = BioMiniProtein(
        "canonical_p1",
        "ACDEFG"
    )

    bm_features = (
        BioMiniProteinAnalyzer(
            bm_protein
        ).to_features()
    )

    bm_sample = BioMiniSample(
        bm_features,
        target=0.5
    )

    bm_dataset = BioMiniDataset()
    bm_dataset.add(
        bm_sample
    )

    print(
        bm_features
    )

    print(
        bm_dataset.to_dataframe()
    )

except ImportError:
    print(
        "BioMini package is not installed in this environment.\n"
        "From the repository root, run:\n"
        "    python -m pip install -e ."
    )

## 21. Educational Version과 Canonical Version의 차이

이번 Notebook은 representation concept에 집중했습니다.

canonical BioMini에서는 추가로 다음을 제공합니다.

```text
schema_version
to_dict()
from_dict()
nested schema validation
```

즉 `FeatureSet`, `Sample`, `Dataset`도 저장하고 복원할 수 있는
versioned scientific data object입니다.

이 내용은 Notebook 07에서 persistence와 함께 본격적으로 다룹니다.

## 22. What We Learned

이번 Notebook의 핵심 흐름:

```text
Protein
 ↓
ProteinAnalyzer
 ↓
numerical values
 ↓
FeatureSet
 ↓
Sample
 ↓
Dataset
 ↓
DataFrame
 ↓
X / y
```

가장 중요한 설계 원칙:

> biological object와 machine-learning representation을 분리한다.

이렇게 하면 domain model은 안정적으로 유지하면서,
representation strategy는 나중에 자유롭게 교체할 수 있습니다.

## 23. Bridge to Notebook 06

이제 우리는 supervised-learning에 필요한:

```text
X
y
```

를 만들 수 있습니다.

다음 단계는 실제 machine-learning model입니다.

Notebook 06에서는:

```text
Dataset
 ↓
StandardScaler
 ↓
Ridge
 ↓
prediction
```

workflow를 구현합니다.

그리고 training과 inference 사이에서
**feature-column order를 반드시 보존해야 하는 이유**를 다룹니다.

다음 주제:

> **Dataset and Machine Learning**

## 24. Exercises

### Exercise 1 — Feature Inspection
다음 Protein의 feature를 생성하십시오.

```text
ACDEFGHIKLMNPQRSTVWY
```

`length`, `molecular_weight`, `hydrophobic_fraction`을 확인하십시오.

### Exercise 2 — Amino-Acid Fraction
`AAAAACCCCC`의 `aa_A_fraction`, `aa_C_fraction`을 확인하십시오.

### Exercise 3 — Invalid FeatureSet
다음을 각각 시도하고 왜 실패하는지 설명하십시오.

```python
FeatureSet("", {"length": 10})
FeatureSet("x", {})
FeatureSet("x", {"length": "ten"})
FeatureSet("x", {"length": float("nan")})
```

### Exercise 4 — Unlabeled Sample
target 없이 `Sample`을 만들고 Dataset에 넣은 뒤 DataFrame을 확인하십시오.

### Exercise 5 — Supervised Dataset
Protein 3개 이상을 이용해 target을 가진 Dataset을 직접 구성하십시오.

### Exercise 6 — New Feature
`ProteinAnalyzer`에 다음 feature를 하나 추가해 보십시오.

```text
charged_fraction
```

어떤 amino acid를 charged로 정의할지 명시하십시오.

### Exercise 7 — Architecture Question
왜 `Protein` class 자체에 `to_ml_vector()`를 넣지 않고
`ProteinAnalyzer → FeatureSet` 구조를 사용하는 것이 더 유연한지 설명하십시오.

### Exercise 8 — Phase II Preview
같은 Protein을 one-hot encoding이나 embedding으로 표현한다면
현재 `FeatureSet` abstraction이 어떻게 확장될 수 있을지 생각해 보십시오.